# SPY Risk Alert Project Pipeline

This cumulative pipeline starts with Stage04 ingestion and Stage05 reproducible storage. By default it reloads the latest retained raw snapshot; set `REFRESH_RAW = True` only for an intentional new Nasdaq acquisition. The source is unadjusted OHLCV and this notebook does not yet construct a return target or make a trading recommendation.

## 1. Project Root, Configuration, and Imports

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project
ROOT = Path.cwd()
if not (ROOT / 'src' / 'ingestion.py').is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / 'project'
        if (project_candidate / 'src' / 'ingestion.py').is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date

from src.config import get_processed_data_dir, get_raw_data_dir, load_env
from src.ingestion import (
    fetch_nasdaq_history, timestamp_utc, validate_spy_history, write_manifest, write_raw_csv
)
from src.storage import get_parquet_engine, read_df, validate_roundtrip, write_df

environment_loaded = load_env()
RAW_DIR = get_raw_data_dir()
PROCESSED_DIR = get_processed_data_dir()
print('working from:', ROOT.name)
print('Environment loaded:', environment_loaded)
print('Raw directory:', RAW_DIR)
print('Processed directory:', PROCESSED_DIR)

working from: project
Environment loaded: True
Raw directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/raw
Processed directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/processed


## 2. Stage04 Raw Snapshot

Normal reruns reuse the latest timestamped Stage04 CSV. An explicit refresh requests a new ten-year Nasdaq snapshot, validates it, and records a new manifest.

In [2]:
REFRESH_RAW = False
SYMBOL = 'SPY'
csv_schema = {
    'open': 'float64', 'high': 'float64', 'low': 'float64',
    'close': 'float64', 'volume': 'int64',
}
raw_candidates = sorted(RAW_DIR.glob('api_nasdaq_spy_daily_*.csv'))

if REFRESH_RAW:
    end_date = date.today()
    start_date = end_date.replace(year=end_date.year - 10)
    spy_raw, source_metadata = fetch_nasdaq_history(
        SYMBOL, start_date=start_date.isoformat(), end_date=end_date.isoformat()
    )
    validation = validate_spy_history(spy_raw)
    snapshot_timestamp = timestamp_utc()
    raw_path = write_raw_csv(
        spy_raw, RAW_DIR, 'api_nasdaq_spy_daily', timestamp=snapshot_timestamp
    )
    manifest_path = write_manifest(
        {
            'path': raw_path, 'dataset': 'SPY daily unadjusted OHLCV',
            'rows': len(spy_raw), 'columns': list(spy_raw.columns),
            'source_metadata': source_metadata, 'validation': validation,
        },
        RAW_DIR / f'ingestion_manifest_{snapshot_timestamp}.json',
    )
    print('Refreshed raw snapshot:', raw_path.name)
    print('Saved manifest:', manifest_path.name)
else:
    if not raw_candidates:
        raise FileNotFoundError('No Stage04 SPY raw snapshot found. Set REFRESH_RAW = True.')
    raw_path = raw_candidates[-1]
    snapshot_timestamp = raw_path.stem.removeprefix('api_nasdaq_spy_daily_')
    spy_raw = read_df(raw_path, parse_dates=['date'], dtype=csv_schema)
    validation = validate_spy_history(spy_raw)
    print('Reused raw snapshot:', raw_path.name)

print('Rows and columns:', validation['shape'])
print('Date range:', validation['date_min'], 'to', validation['date_max'])
spy_raw.head()

Reused raw snapshot: api_nasdaq_spy_daily_20260907-143336.csv
Rows and columns: [2512, 6]
Date range: 2016-09-07 to 2026-09-04


,date,open,high,low,close,volume
0,2016-09-07,218.84,219.2200,218.30,219.01,76302150
1,2016-09-08,218.62,218.9400,218.15,218.51,73855230
2,2016-09-09,216.97,217.0300,213.25,213.28,220309300
3,2016-09-12,212.39,216.8100,212.31,216.34,167653400
4,2016-09-13,214.84,215.1499,212.50,213.23,182323200


## 3. Stage05 Processed Storage and Round-Trip Validation

The Parquet file is a typed, reproducible representation of the named raw snapshot. It is not a cleaning or feature-engineering step.

In [3]:
processed_path = PROCESSED_DIR / f'spy_ohlcv_nasdaq_{snapshot_timestamp}.parquet'
write_df(spy_raw, processed_path)
spy_processed = read_df(processed_path)
roundtrip = validate_roundtrip(
    spy_raw, spy_processed,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not roundtrip['passed']:
    raise ValueError(f'Parquet round-trip validation failed: {roundtrip}')

print('Parquet engine:', get_parquet_engine())
print('Saved processed file:', processed_path.name)
print('Round-trip validation passed:', roundtrip['passed'])
print('CSV bytes:', raw_path.stat().st_size)
print('Parquet bytes:', processed_path.stat().st_size)

Parquet engine: pyarrow
Saved processed file: spy_ohlcv_nasdaq_20260907-143336.parquet
Round-trip validation passed: True
CSV bytes: 120981
Parquet bytes: 102731


## Sources, Storage, Assumptions, and Risks

- Source and raw validation rules: `docs/data_sources.md`.
- Storage convention and lineage: `docs/data_storage.md`.
- `data/raw/` is never overwritten; `data/processed/` is reproducible from a named raw snapshot and code.
- The source remains unadjusted OHLCV. Corporate-action handling and the high-volatility target are deferred to preprocessing.
- Future stages extend this same notebook with preprocessing, EDA, features, modeling, and reporting.